In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib as pl

In [ ]:
root_path = pl.Path.cwd().parent.parent
data_path = root_path / "data" / "intermed"

In [ ]:
data_review = pd.read_csv(data_path/"googleplaystore_user_reviews_clean.csv")
data_apps = pd.read_csv(data_path/"googleplaystore_clean.csv")

In [ ]:
merged_data = pd.merge(left=data_apps,right=data_review,how="inner",on=["App"])

In [ ]:

gp_data = merged_data.groupby("Genres").agg({
    "Sentiment_Polarity": ["mean", "std"], 
    "Rating": "mean"
}).dropna() 


gp_data.columns = ["Mean_Sentiment", "Std_Sentiment", "Mean_Rating"]

gp_data["Rating_Norm"] = gp_data["Mean_Rating"] / 5.0
gp_data["Sentiment_Norm"] = (gp_data["Mean_Sentiment"] + 1) / 2.0

gp_data["Gap_Normalized"] = abs(gp_data["Sentiment_Norm"] - gp_data["Rating_Norm"])

gp_data["Score_Oportunidade"] = gp_data["Gap_Normalized"] * gp_data["Std_Sentiment"]

top_targets = gp_data.sort_values(by="Score_Oportunidade", ascending=False)
print("=== TOP 5: A TEMPESTADE PERFEITA (OCEANO AZUL) ===")
print(top_targets[["Mean_Rating", "Std_Sentiment", "Gap_Normalized", "Score_Oportunidade"]].head(5))


In [ ]:
plt.figure(figsize=(16,4))
plt.title("Top 5 Gêneros com Maiores Scores",fontsize=16, weight='bold')
sns.barplot(data=top_targets.head(5),y="Score_Oportunidade",x=top_targets.head(5).index,palette="magma")




In [ ]:
plt.figure(figsize=(16, 8))


sns.scatterplot(
    data=gp_data, 
    x="Std_Sentiment",       
    y="Gap_Normalized",      
    size="Score_Oportunidade", 
    sizes=(50, 500), 
    hue="Score_Oportunidade", 
    palette="magma",         
    alpha=0.85,
    legend=False             
)


limite_x = gp_data["Std_Sentiment"].median()
limite_y = gp_data["Gap_Normalized"].median()

plt.axvline(limite_x, color='gray', linestyle='--', alpha=0.5, zorder=0)
plt.axhline(limite_y, color='gray', linestyle='--', alpha=0.5, zorder=0)


for genero, row in top_targets.head(5).iterrows():
    plt.annotate(
        genero, 
        xy=(row['Std_Sentiment'], row['Gap_Normalized']),
        xytext=(row['Std_Sentiment'] + 0.015, row['Gap_Normalized'] + 0.015),
        fontsize=11, weight='bold', color='darkred',
        arrowprops=dict(arrowstyle="-", color='gray', alpha=0.6)
    )


plt.text(limite_x + 0.01, gp_data["Gap_Normalized"].max() * 0.95, "A Tempestade Perfeita\n(Métricas Falsas + Alta Guerra)", color='purple', weight='bold', fontsize=11, alpha=0.8)
plt.text(gp_data["Std_Sentiment"].min() * 1.05, gp_data["Gap_Normalized"].max() * 0.95, "Mentira Unânime\n(Bots / Sarcasmo)", color='firebrick', weight='bold', fontsize=11, alpha=0.6)
plt.text(limite_x + 0.01, gp_data["Gap_Normalized"].min() * 1.05, "Guerra Transparente\n(Ame ou Odeie Claro)", color='darkorange', weight='bold', fontsize=11, alpha=0.6)


plt.xlabel("Nível de Caos e Polarização (Desvio Padrão do Texto)", fontsize=12)
plt.ylabel("Tamanho da Discrepância (Gap entre Nota e Sentimento)", fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
gp_data